<a href="https://colab.research.google.com/github/DaveWells-C/Soccer-Analytics/blob/main/notebooks/Day_8_OCSC_ASA_xGoals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q itscalledsoccer

In [11]:
import pandas as pd
from itscalledsoccer.client import AmericanSoccerAnalysis


# 1. Initialize the API client (takes a few seconds to load league metadata)
asa = AmericanSoccerAnalysis()

# 2. Look up Orange County SC's official team record in the USL Championship ("uslc")
ocsc_team = asa.get_teams(leagues="uslc", names="Orange County")
display(ocsc_team)

# 3. Pull all USL Championship player profiles so we can match player names later
uslc_players = asa.get_players(leagues="uslc")
print(f"Loaded {len(uslc_players)} USL Championship player profiles.")

team_id,team_name,team_short_name,team_abbreviation,competition
str,str,str,str,str
"""kRQaeY8QKZ""","""Orange County SC""","""Orange County""","""OC""","""uslc"""


Loaded 3817 USL Championship player profiles.


In [12]:
# 1. Pull 2026 Orange County SC player xGoals data
ocsc_xg_raw = asa.get_player_xgoals(
    leagues="uslc",
    team_names="Orange County",
    season_name="2026"
)

# Convert to pandas if the library returned a Polars DataFrame
if not isinstance(ocsc_xg_raw, pd.DataFrame):
    ocsc_xg_raw = ocsc_xg_raw.to_pandas()
if not isinstance(uslc_players, pd.DataFrame):
    uslc_players = uslc_players.to_pandas()

# 2. Merge player names onto the xGoals stats using player_id
ocsc_xg = pd.merge(
    ocsc_xg_raw,
    uslc_players[["player_id", "player_name"]],
    on="player_id",
    how="left"
)

# 3. Inspect the columns and preview the top 8 rows sorted by xGoals
print("Available xG columns:", ocsc_xg.columns.tolist())
display(ocsc_xg[["player_name", "general_position", "minutes_played", "shots", "goals", "xgoals", "xassists"]].sort_values(by="xgoals", ascending=False).round(2).head(8))

Available xG columns: ['xgoals', 'xplace', 'xassists', 'key_passes', 'goals_plus_primary_assists', 'primary_assists_minus_xassists', 'player_id', 'goals', 'goals_minus_xgoals', 'points_added', 'minutes_played', 'team_id', 'xgoals_plus_xassists', 'shots_on_target', 'xpoints_added', 'general_position', 'primary_assists', 'shots', 'player_name']


,player_name,general_position,minutes_played,shots,goals,xgoals,xassists
9,Yaniv Bazini,ST,1876,46,10,8.7796,1.7153
2,Lyam MacKinnon,W,2015,51,4,7.7634,2.3663
3,Ousmane Sylla,AM,2163,33,6,6.0047,3.2104
19,Christopher Hegardt,W,2561,43,5,4.1377,4.9804
5,Nico Benalcazar,FB,2483,15,3,2.1318,1.132
11,Stephen Kelly,CM,2597,26,3,1.6174,4.2498
28,Garrison Tubbs,CB,2138,10,2,1.4753,0.0955
18,Tom Brewitt,CB,2831,19,1,1.3109,0.1277


In [13]:
uslc_team_xg = asa.get_team_xgoals(leagues="uslc", season_name="2026")
uslc_teams = asa.get_teams(leagues="uslc")

if not isinstance(uslc_team_xg, pd.DataFrame):
    uslc_team_xg = uslc_team_xg.to_pandas()
if not isinstance(uslc_teams, pd.DataFrame):
    uslc_teams = uslc_teams.to_pandas()

team_xg = pd.merge(
    uslc_team_xg,
    uslc_teams,
    on="team_id",
    how="left"
)

ocsc_xg = team_xg[team_xg["team_name"].str.contains("Orange County", na=False)]
display(ocsc_xg[["team_name", "goals_for", "xgoals_for", "goals_against", "xgoals_against", "xgoal_difference"]].round(2))


,team_name,goals_for,xgoals_for,goals_against,xgoals_against,xgoal_difference
23,Orange County SC,37,39.63,49,54.33,-14.7


In [14]:
# 1. Rank all 2026 USL Championship clubs from best to worst xgoal_difference
team_xg["xGD_Rank"] = team_xg["xgoal_difference"].rank(ascending=False).astype(int)

# 2. Sort the full league table by xgoal_difference descending
league_xg_table = (
    team_xg[["xGD_Rank", "team_name", "count_games", "goals_for", "xgoals_for", "goals_against", "xgoals_against", "xgoal_difference"]]
    .sort_values(by="xgoal_difference", ascending=False)
    .round(2)
)

# 3. Display the Top 5 clubs in the league for context
print("--- Top 5 USL Championship Clubs by xG Difference ---")
display(league_xg_table.head(5))

# 4. Pull out Orange County SC's exact league rank row
print("\n--- Orange County SC League Standing ---")
ocsc_rank_row = league_xg_table[league_xg_table["team_name"].str.contains("Orange County", na=False)]
display(ocsc_rank_row)

--- Top 5 USL Championship Clubs by xG Difference ---


,xGD_Rank,team_name,count_games,goals_for,xgoals_for,goals_against,xgoals_against,xgoal_difference
4,1,Tampa Bay Rowdies,31,51,58.90,26,36.17,22.73
9,2,Colorado Springs Switchbacks FC,32,43,58.15,40,38.21,19.94
24,3,Lexington SC,31,56,60.51,39,42.82,17.68
0,4,Charleston Battery,33,59,58.89,40,41.44,17.46
16,5,Louisville City FC,34,56,61.09,44,43.86,17.23



--- Orange County SC League Standing ---


,xGD_Rank,team_name,count_games,goals_for,xgoals_for,goals_against,xgoals_against,xgoal_difference
23,20,Orange County SC,30,37,39.63,49,54.33,-14.7


# Orange County SC — 2026 USL Championship Expected Goals (`xG`) Analysis
**Tools:** Python, `pandas`, `itscalledsoccer` API, Google Colab | **Source:** American Soccer Analysis

### What This Notebook Does
* Connects to the American Soccer Analysis API (`uslc`, 2026 season).
* Pulls Orange County SC player `xGoals` and `xAssists` merged with player names.
* Ranks Orange County SC's team `xGoal Difference` (`xgoals_for - xgoals_against`) against the full 2026 USL Championship table.